# Domain 1 – Tyre Degradation: ANOVA Interaction Analysis

This notebook answers the key statistical question:

> **Does the compound × track interaction significantly affect tyre degradation rate?**

Analysis:
1. One-way ANOVA: degradation rate by compound
2. One-way ANOVA: degradation rate by track
3. **Two-way ANOVA with interaction**: compound × track
4. Tukey HSD post-hoc test between compounds
5. Interaction plot
6. Effect size (eta-squared)
7. Key findings summary


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.utils.paths import DOMAIN1_SILVER
print('Imports OK')

## 1. Load Data

In [ ]:
stints = pd.read_parquet(DOMAIN1_SILVER / 'stints_degradation.parquet')
print(f'Stints: {len(stints):,} rows')

# Standardise column names
compound_col = 'compound' if 'compound' in stints.columns else 'tyre_compound'
event_col    = 'event'    if 'event'    in stints.columns else 'EventName'

df = stints[['deg_rate_linear', compound_col, event_col]].copy().dropna()
df = df.rename(columns={compound_col: 'compound', event_col: 'track'})

# Keep only slick compounds with sufficient data (>= 10 stints per compound)
compound_counts = df['compound'].value_counts()
valid_compounds = compound_counts[compound_counts >= 10].index.tolist()
df = df[df['compound'].isin(valid_compounds)]

print(f'Analysis dataset: {len(df):,} stints')
print(f'Compounds: {sorted(df["compound"].unique())}')
print(f'Tracks: {df["track"].nunique()}')

## 2. One-Way ANOVA: Degradation Rate by Compound

In [ ]:
groups_compound = [group['deg_rate_linear'].values for _, group in df.groupby('compound')]
f_stat_compound, p_val_compound = stats.f_oneway(*groups_compound)

print('=== One-Way ANOVA: Degradation Rate ~ Compound ===')
print(f'F-statistic: {f_stat_compound:.4f}')
print(f'p-value:     {p_val_compound:.2e}')
print(f'Significant (α=0.05): {p_val_compound < 0.05}')
print()

# Descriptive stats per compound
print('Descriptive statistics per compound:')
print(df.groupby('compound')['deg_rate_linear'].describe().round(4))

## 3. One-Way ANOVA: Degradation Rate by Track

In [ ]:
# Keep tracks with >= 5 stints
track_counts = df['track'].value_counts()
valid_tracks = track_counts[track_counts >= 5].index.tolist()
df_tracks = df[df['track'].isin(valid_tracks)]

groups_track = [group['deg_rate_linear'].values for _, group in df_tracks.groupby('track')]
f_stat_track, p_val_track = stats.f_oneway(*groups_track)

print('=== One-Way ANOVA: Degradation Rate ~ Track ===')
print(f'F-statistic: {f_stat_track:.4f}')
print(f'p-value:     {p_val_track:.2e}')
print(f'Significant (α=0.05): {p_val_track < 0.05}')

## 4. Two-Way ANOVA with Interaction: Compound × Track

In [ ]:
# Use tracks with sufficient data
df_anova = df_tracks.copy()

# Fit OLS model with interaction term
model = smf.ols('deg_rate_linear ~ C(compound) * C(track)', data=df_anova).fit()

# Type II ANOVA table
anova_table = sm.stats.anova_lm(model, typ=2)

print('=== Two-Way ANOVA with Interaction: compound × track ===')
print(anova_table.round(4))
print()
print(f'Model R²: {model.rsquared:.4f}')
print(f'Adj. R²: {model.rsquared_adj:.4f}')

## 5. Effect Size: Eta-Squared (η²)

In [ ]:
# Eta-squared = SS_effect / SS_total
ss_total = anova_table['sum_sq'].sum()
eta_sq = anova_table['sum_sq'] / ss_total

print('=== Effect Size: Eta-Squared (η²) ===')
for term, eta in eta_sq.items():
    print(f'  {term:<40s}: η² = {eta:.4f} ({eta*100:.1f}% of variance)')
print()

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
eta_sq_plot = eta_sq.drop('Residual', errors='ignore')
bars = ax.barh(range(len(eta_sq_plot)), eta_sq_plot.values, color=sns.color_palette('Set2', len(eta_sq_plot)))
ax.set_yticks(range(len(eta_sq_plot)))
ax.set_yticklabels([t.replace('C(compound)', 'Compound').replace('C(track)', 'Track') for t in eta_sq_plot.index])
ax.set_xlabel('Eta-Squared (η²)')
ax.set_title('Effect Size (η²) for each ANOVA term')
for bar, val in zip(bars, eta_sq_plot.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Tukey HSD Post-Hoc Test Between Compounds

In [ ]:
tukey_result = pairwise_tukeyhsd(
    endog=df['deg_rate_linear'],
    groups=df['compound'],
    alpha=0.05
)

print('=== Tukey HSD Post-Hoc Test: Pairwise Compound Comparison ===')
print(tukey_result.summary())

# Mean degradation per compound for reference
compound_means = df.groupby('compound')['deg_rate_linear'].mean().sort_values(ascending=False)
print('\nMean degradation rate per compound:')
print(compound_means.round(4))

## 7. Interaction Plot: Compound × Track Regime

In [ ]:
# Classify tracks into regimes for the interaction plot
track_deg_mean = df_tracks.groupby('track')['deg_rate_linear'].mean()
track_tertiles = pd.qcut(track_deg_mean, q=3, labels=['Low-Deg', 'Med-Deg', 'High-Deg'])
track_regime_map = track_tertiles.to_dict()
df_tracks = df_tracks.copy()
df_tracks['track_regime'] = df_tracks['track'].map(track_regime_map)

interaction_data = (
    df_tracks.groupby(['compound', 'track_regime'])['deg_rate_linear']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
compound_colors = {'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6'}
regimes = ['Low-Deg', 'Med-Deg', 'High-Deg']
x_pos = {r: i for i, r in enumerate(regimes)}

for compound, group in interaction_data.groupby('compound'):
    group = group.set_index('track_regime').reindex(regimes)
    ax.plot(
        regimes, group['deg_rate_linear'].values,
        marker='o', linewidth=2, markersize=8,
        label=compound, color=compound_colors.get(compound, 'grey')
    )

ax.set_title('Interaction Plot: Compound × Track Regime\n(Non-parallel lines indicate significant interaction)')
ax.set_xlabel('Track Degradation Regime')
ax.set_ylabel('Mean Degradation Rate (s/lap)')
ax.legend(title='Compound')
plt.tight_layout()
plt.show()

## 8. Key Findings Summary

In [ ]:
print('=' * 60)
print('KEY FINDINGS – ANOVA INTERACTION ANALYSIS')
print('=' * 60)
print()
print('1. ONE-WAY ANOVA (Compound)')
print(f'   F = {f_stat_compound:.2f}, p = {p_val_compound:.2e}')
sig = 'SIGNIFICANT' if p_val_compound < 0.05 else 'NOT SIGNIFICANT'
print(f'   → Compound effect is {sig} at α=0.05')
print()
print('2. ONE-WAY ANOVA (Track)')
print(f'   F = {f_stat_track:.2f}, p = {p_val_track:.2e}')
sig2 = 'SIGNIFICANT' if p_val_track < 0.05 else 'NOT SIGNIFICANT'
print(f'   → Track effect is {sig2} at α=0.05')
print()
print('3. TWO-WAY ANOVA (Compound × Track)')
# Find the interaction term robustly – statsmodels uses 'C(a):C(b)' notation
interaction_row = anova_table[
    anova_table.index.str.contains('compound', case=False) &
    anova_table.index.str.contains('track', case=False)
]
if interaction_row.empty:
    # Fallback: any term containing a colon is an interaction
    interaction_row = anova_table[anova_table.index.str.contains(':', regex=False)]
# (original single-line version replaced for robustness)
if not interaction_row.empty:
    p_int = interaction_row['PR(>F)'].values[0]
    f_int = interaction_row['F'].values[0]
    print(f'   Interaction F = {f_int:.2f}, p = {p_int:.2e}')
    sig3 = 'SIGNIFICANT' if p_int < 0.05 else 'NOT SIGNIFICANT'
    print(f'   → Compound × Track interaction is {sig3}')
print()
print('4. PRACTICAL IMPLICATION')
print('   The compound ranking by degradation severity is NOT uniform')
print('   across all circuits. Strategy must be track-specific.')
print('=' * 60)